# Bài tập 2 — Làm sạch lần 1: hợp nhất hai nguồn → `df_raw.csv`
**Phụ trách:** Nguyễn Bá Công, Hà Trọng Hữu Duy

Toàn bộ logic nằm trong `src/data/lam_sach_du_lieu.py` (hàm `tao_df_raw`) để chạy lại được bằng lệnh
`python -m src.data.lam_sach_du_lieu`. Notebook này chạy hàm đó và kiểm tra kết quả.

| Bước | Việc làm |
|---|---|
| 1 | Đọc CafeLand + batdongsan, ánh xạ về schema chung |
| 2 | Tách tin bán / tin cho thuê |
| 3 | Lọc TP.HCM |
| 4 | Chuẩn hóa sơ bộ nhãn loại hình |
| 5 | Bỏ tin thiếu giá hoặc diện tích |
| 6 | Lọc ngoại lai thô (ngưỡng cứng + IQR×3) |
| 7 | Khử trùng URL và trùng chéo nguồn |
| 8 | Tạo đơn giá `gia_moi_m2_trieu` |
| 9 | Trích quận/huyện từ mô tả, bỏ tin không rõ quận |

In [1]:
import sys
from pathlib import Path

# Thư mục gốc dự án = thư mục cha của notebooks/
GOC = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(GOC))

import re
import numpy as np
import pandas as pd
from src.utils import doc_config

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
cfg = doc_config()
THU_MUC_THO = GOC / cfg['duong_dan']['du_lieu_tho']
THU_MUC_TRUNG_GIAN = GOC / cfg['duong_dan']['du_lieu_trung_gian']
THU_MUC_SACH = GOC / cfg['duong_dan']['du_lieu_sach']
print('Thư mục gốc:', GOC)

Thư mục gốc: /home/claude/repo


In [2]:
from src.data.lam_sach_du_lieu import tao_df_raw

df_raw = tao_df_raw(luu=True)   # ghi data/interim/df_raw_chay_lai.csv


BƯỚC 1: ĐỌC DỮ LIỆU TỪ 3 NGUỒN


  [CafeLand]    đọc được 9,883 dòng


  [batdongsan]  đọc được 30,149 dòng
  ⏭  Bỏ qua [chotot]: không nằm trong cfg['nguon_su_dung']

TÓM TẮT: SAU KHI GỘP 3 NGUỒN
Kích thước      : 40,032 dòng × 9 cột
Trùng lặp toàn bộ: 0 dòng
Thiếu dữ liệu   :
    quan_huyen                      40,032 (100.0%)
    loai_hinh                        8,776 ( 21.9%)
    dien_tich_m2                     1,295 (  3.2%)
    gia_ban                            914 (  2.3%)
    mo_ta_dac_diem                     422 (  1.1%)

BƯỚC 2: TÁCH TIN BÁN / TIN CHO THUÊ


  Tin BÁN : 38,582
  Tin THUÊ: 1,450 (sẽ loại bỏ)

BƯỚC 3: LỌC PHẠM VI TP.HCM
  Lọc TP.HCM: 38,582 -> 36,339 dòng (loại 2,243)

BƯỚC 4: CHUẨN HÓA NHÃN LOẠI HÌNH
  Số nhãn sau chuẩn hóa: 12
    Nhà rieng              9,445
    Đất                    6,477
    căn hộ                 5,401
    Nhà phố                2,458
    Biệt thự               2,362
    Nhà mặt tiền           1,936
    Shophouse                272
    Nhà hàng - Khách sạn      66
    Kho - Nhà xưởng           57
    Bán đất nông, lâm nghiệp      14
    Bán nhà hàng - Khách sạn      14
    Bán kho, nhà xưởng        11
  Số dòng thiếu loai_hinh: 7,826

BƯỚC 5: LOẠI DÒNG THIẾU GIÁ HOẶC DIỆN TÍCH
  36,339 -> 34,736 (loại 1,603)

BƯỚC 6: LỌC NGOẠI LAI
  Lọc ngưỡng cứng: 34,736 -> 34,222
  Lọc IQR (hệ số 3.0): -> 29,799
  Tổng cộng loại: 4,937 dòng ngoại lai

BƯỚC 7: LOẠI TRÙNG LẶP
  Loại trùng URL nội bộ: 29,799 -> 29,799


  Loại trùng chéo nguồn: 29,799 -> 29,532 (loại 267)

BƯỚC 8: TẠO BIẾN PHÁI SINH
  Đã tạo cột: gia_moi_m2_trieu

KẾT QUẢ CUỐI CÙNG

TÓM TẮT: DỮ LIỆU SẠCH
Kích thước      : 29,532 dòng × 10 cột
Trùng lặp toàn bộ: 0 dòng
Thiếu dữ liệu   :
    quan_huyen                      29,532 (100.0%)
    loai_hinh                        6,703 ( 22.7%)
    mo_ta_dac_diem                       4 (  0.0%)

Phân bố theo nguồn:
    batdongsan       23,370 ( 79.1%)
    cafeland          6,162 ( 20.9%)

BƯỚC 9: BỔ SUNG QUẬN/HUYỆN TỪ MÔ TẢ


  Bỏ tin không xác định được quận: 29,532 -> 11,888 (loại 17,644)



✅ Đã lưu: /home/claude/repo/data/interim/df_raw_chay_lai.csv  (11,888 dòng × 10 cột)


## Đối chiếu với `df_raw.csv` nhóm đã chốt
`data/interim/df_raw.csv` là bản dùng cho Project Charter và Data Dictionary. Lần chạy gốc khử trùng chéo nguồn với cách sắp xếp không ổn định, nên chạy lại cho **cùng số dòng và cùng phân bố nguồn**, chỉ khác bản đại diện của một số ít tin trùng. Các bước sau dùng bản đã chốt.

In [3]:
df_raw_chot = pd.read_csv(THU_MUC_TRUNG_GIAN / 'df_raw.csv')
print('Số dòng  – chốt:', len(df_raw_chot), '| chạy lại:', len(df_raw))
print('Nguồn    – chốt:', df_raw_chot.nguon_du_lieu.value_counts().to_dict(), '| chạy lại:', df_raw.nguon_du_lieu.value_counts().to_dict())
chung = df_raw.url.isin(df_raw_chot.url).sum()
print(f'URL trùng khớp: {chung:,}/{len(df_raw):,} ({chung/len(df_raw):.2%})')

Số dòng  – chốt: 11888 | chạy lại: 11888
Nguồn    – chốt: {'batdongsan': 8374, 'cafeland': 3514} | chạy lại: {'batdongsan': 8374, 'cafeland': 3514}
URL trùng khớp: 11,797/11,888 (99.23%)


## Kiểm tra kết quả

In [4]:
print('Kích thước:', df_raw.shape)
print('Giá <= 0      :', (df_raw['gia_ban'] <= 0).sum())
print('Diện tích <= 0:', (df_raw['dien_tich_m2'] <= 0).sum())
print('URL trùng     :', df_raw['url'].duplicated().sum())
print('Dòng trùng    :', df_raw.duplicated().sum())
print('Tỉnh/thành    :', df_raw['tinh_thanh'].unique())
print('Số quận/huyện :', df_raw['quan_huyen'].nunique())
print()
print('Ô trống theo cột:'); print(df_raw.isna().sum()[lambda s: s > 0])

Kích thước: (11888, 10)
Giá <= 0      : 0
Diện tích <= 0: 0
URL trùng     : 0
Dòng trùng    : 0
Tỉnh/thành    : ['TP. Hồ Chí Minh']
Số quận/huyện : 20

Ô trống theo cột:
loai_hinh    2808
dtype: int64


In [5]:
phan_bo = df_raw['nguon_du_lieu'].value_counts()
pd.DataFrame({'so_tin': phan_bo, 'ty_le_%': (phan_bo / len(df_raw) * 100).round(2)})

,so_tin,ty_le_%
nguon_du_lieu,,
batdongsan,8374,70.44
cafeland,3514,29.56


In [6]:
df_raw[['gia_ban', 'dien_tich_m2', 'gia_moi_m2_trieu']].describe().round(2)

,gia_ban,dien_tich_m2,gia_moi_m2_trieu
count,11888.00,11888.00,11888.00
mean,9568.24,81.01,126.53
std,7653.86,45.19,88.43
min,109.00,10.00,1.18
25%,4700.00,51.00,74.14
50%,6990.00,70.00,107.69
75%,11900.00,100.00,153.77
max,44000.00,270.00,2500.00


## Nhận xét
- Bản chạy lại và bản chốt cùng 11.888 dòng, cùng 3.514 tin CafeLand / 8.374 tin batdongsan.
- `df_raw` đạt yêu cầu tối thiểu (≥ 5.000 dòng, ≥ 8 biến), 100% TP.HCM, URL duy nhất.
- Bước 9 loại nhiều tin nhất: phần lớn tin batdongsan không ghi quận trong mô tả nên không phục vụ được câu hỏi Q2.
- **Còn trống:** `loai_hinh` (2.809 dòng ở bản chốt, toàn bộ thuộc batdongsan) → xử lý ở `02_lam_sach_du_lieu_2.ipynb`.
- `gia_moi_m2_trieu` = giá ÷ diện tích → chỉ dùng để mô tả, **phải bỏ** trước khi mô hình hóa.
- Code hiện dùng `kind="stable"` nên từ nay mỗi lần chạy lại cho cùng một tập dòng.

### Ghi chú sử dụng công cụ AI
Theo mục IV của đề cương, nhóm ghi rõ phần có AI hỗ trợ:
- **Claude (Anthropic)** hỗ trợ: rà soát tính tái lập, gom các bước làm sạch thành hàm trong `src/`, đổi đường dẫn tuyệt đối sang tương đối, viết bảng kiểm toán số dòng bị loại, tái tạo bước lọc APE để kiểm chứng.
- Logic xử lý (ngưỡng lọc, cách điền khuyết, regex trích đặc trưng) do nhóm thiết kế và giải thích được; mọi con số trong notebook được sinh ra khi chạy lại, không nhập tay.